In [0]:
# PREPRAR DATOS PARA ML
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [0]:
df_ml = spark.table("default.clientes_credito_limpio")

In [0]:
#DEFINIMOS LAS COLUMNAS 
categorical_cols = [
    "estado_civil",
    "nivel_educativo",
    "situacion_laboral",
    "cliente_recurrente"
]

numeric_cols = [
    "edad",
    "numero_dependientes",
    "antiguedad_laboral_meses",
    "ingreso_mensual",
    "deuda_total",
    "cuota_mensual_total",
    "ratio_deuda_ingreso",
    "numero_creditos_activos",
    "atrasos_previos",
    "max_dias_atraso",
    "score_crediticio",
    "monto_solicitado",
    "plazo_meses",
    "cuota_credito_solicitado",
    "creditos_pagados_correctamente"
]

target_col = "resultado_pago"

In [0]:
#TRANSFORMAR VARIABLE OBJETIVO
label_indexer = StringIndexer(
    inputCol=target_col,
    outputCol="label"
)

In [0]:
# TRANSFORMAR VARIABLES CATEGORICAS
indexers = [
    StringIndexer(
        inputCol=columna,
        outputCol=columna + "_idx",
        handleInvalid="keep"
    )
    for columna in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=columna + "_idx",
        outputCol=columna + "_vec"
    )
    for columna in categorical_cols
]

In [0]:
# CREAR VECTOR DE CARACTERISTICAS
feature_cols = numeric_cols + [columna + "_vec" for columna in categorical_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

In [0]:
# DIVIDIMOS NTRENAMIENTO Y PRUEBA
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=123)

print("Registros entrenamiento:", train_df.count())
print("Registros prueba:", test_df.count())

Registros entrenamiento: 16054
Registros prueba: 3946


In [0]:
#CREAMOS EL MODELO  DECISION TREE
decision_tree = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=5,
    minInstancesPerNode=20,
    seed=123
)

In [0]:
#CREAR PIPELINE
pipeline = Pipeline(stages=[
    label_indexer
] + indexers + encoders + [
    assembler,
    decision_tree
])

In [0]:
# ENTRENAMOS EL MODELO
model = pipeline.fit(train_df)

In [0]:
#GENERAR PREDICCIONES

predictions = model.transform(test_df)

display(
    predictions.select(
        "cliente_id",
        "resultado_pago",
        "label",
        "prediction",
        "probability"
    )
)

cliente_id,resultado_pago,label,prediction,probability
4,mal_pagador,1.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0""]}"
7,buen_pagador,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9687251944753136"",""0.03127480552468646""]}"
9,buen_pagador,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9687251944753136"",""0.03127480552468646""]}"
13,buen_pagador,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9687251944753136"",""0.03127480552468646""]}"
17,mal_pagador,1.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0""]}"
18,mal_pagador,1.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8271954674220963"",""0.17280453257790368""]}"
22,mal_pagador,1.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0""]}"
24,buen_pagador,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8392857142857143"",""0.16071428571428573""]}"
49,mal_pagador,1.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9687251944753136"",""0.03127480552468646""]}"
52,mal_pagador,1.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.29133858267716534"",""0.7086614173228346""]}"


In [0]:
#EVALUANDO EL MODELO

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = accuracy_evaluator.evaluate(predictions)

print("Accuracy:", accuracy)

Accuracy: 0.9386720729853015


In [0]:
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

precision = precision_evaluator.evaluate(predictions)

print("Precision:", precision)

Precision: 0.9384751440014084


In [0]:
recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

recall = recall_evaluator.evaluate(predictions)

print("Recall:", recall)

Recall: 0.9386720729853015


In [0]:
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

f1 = f1_evaluator.evaluate(predictions)

print("F1 Score:", f1)

F1 Score: 0.938296702504009


In [0]:
#MATRIZ DE CONFUCION
confusion_matrix = predictions.groupBy("label", "prediction").count()

display(confusion_matrix)

label,prediction,count
1.0,1.0,1191
1.0,0.0,153
0.0,1.0,89
0.0,0.0,2513


In [0]:
#CRUCE CON VALOR ORIGINAL
display(
    predictions.groupBy("resultado_pago", "prediction").count()
)

resultado_pago,prediction,count
buen_pagador,0.0,2513
mal_pagador,1.0,1191
mal_pagador,0.0,153
buen_pagador,1.0,89


In [0]:
#PARA VER QUE VARIABLES FUERON MAS IMPORTANTE EN EL MODELO
dt_model = model.stages[-1]

print(dt_model.featureImportances)

(29,[3,6,8,9,10,14],[0.17012535155907427,0.08140883514661684,0.20110874926735092,0.19956580682347158,0.2563773352549377,0.0914139219485487])


In [0]:
import pandas as pd

importances = dt_model.featureImportances.toArray()

feature_importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": importances[:len(feature_cols)]
}).sort_values(by="importance", ascending=False)

display(feature_importance_df)

feature,importance
score_crediticio,0.2563773352549377
atrasos_previos,0.20110874926735092
max_dias_atraso,0.19956580682347158
ingreso_mensual,0.17012535155907427
creditos_pagados_correctamente,0.0914139219485487
ratio_deuda_ingreso,0.08140883514661684
plazo_meses,0.0
situacion_laboral_vec,0.0
nivel_educativo_vec,0.0
estado_civil_vec,0.0


In [0]:
df_etl.groupBy("resultado_pago").count().show()

+---------+
|catalog  |
+---------+
|samples  |
|system   |
|workspace|
+---------+



In [0]:
#GUARDAMOS EL MODELO
ruta_modelo = "/Volumes/workspace/default/data_clientes/modelo_arbol_buen_pagador"
model.write().overwrite().save(ruta_modelo)

In [0]:
#CARGAMOS EL MODELO
from pyspark.ml import PipelineModel

modelo_cargado = PipelineModel.load("/Volumes/workspace/default/data_clientes/modelo_arbol_buen_pagador")

In [0]:
#PROBAMOS EL MODELO

nuevo_cliente = spark.createDataFrame([
    (
        999999,
        34,
        "Casado",
        "Universitario",
        2,
        "Dependiente",
        48,
        30000000,
        80,
        900,
        0.5,
        2,
        0,
        0,
        720,
        5000,
        12,
        47000.50,
        "No",
        3,
        "buen_pagador"
    )
], schema=df_ml.schema)

prediccion_nuevo = modelo_cargado.transform(nuevo_cliente)

display(
    prediccion_nuevo.select(
        "cliente_id",
        "prediction",
        "probability"
    )
)

cliente_id,prediction,probability
999999,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9687251944753136"",""0.03127480552468646""]}"
